In [12]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

Tool Creation

In [9]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factoor between base currency and target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/b16eb56db51ba5085f273694/pair/{base_currency}/{target_currency}'
    response = requests.get(url)

    return response.json()

In [10]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1794182401,
 'time_last_update_utc': 'Mon, 09 Nov 2026 00:00:01 +0000',
 'time_next_update_unix': 1794268801,
 'time_next_update_utc': 'Tue, 10 Nov 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.0567}

In [6]:
@tool
def convert(base_currency_value: int, conversion_rate: float) -> float:
    """
    Given a currency conversion rate this function calculates the target currency value from a given base currency value
    """
    return base_currency_value * conversion_rate

In [7]:
convert.invoke({'base_currency_value': 100, 'conversion_rate': 94.0567})

9405.67

Tool Binding

In [8]:
llm = ChatGroq(model='qwen/qwen3.8-27b')

In [11]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [13]:
messages = [HumanMessage('what is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR')]

In [15]:
messages

[HumanMessage(content='what is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={})]

In [16]:
ai_message = llm_with_tools.invoke(messages)
ai_message

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'f2rk3p9v5', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 407, 'total_tokens': 448, 'completion_time': 0.118374914, 'completion_tokens_details': None, 'prompt_time': 0.031669006, 'prompt_tokens_details': None, 'queue_time': 0.125454004, 'total_time': 0.15004392}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1c0a2-6a49-7741-8427-12a1e91cbc37-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'f2rk3p9v5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 407, 'output_tokens': 41, 'total_tokens': 448})

In [18]:
ai_message.tool_calls[0]

{'name': 'get_conversion_factor',
 'args': {'base_currency': 'USD', 'target_currency': 'INR'},
 'id': 'f2rk3p9v5',
 'type': 'tool_call'}

In [19]:
conversion_factor = get_conversion_factor.invoke(ai_message.tool_calls[0])
conversion_factor

ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1794182401, "time_last_update_utc": "Mon, 09 Nov 2026 00:00:01 +0000", "time_next_update_unix": 1794268801, "time_next_update_utc": "Tue, 10 Nov 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 94.0567}', name='get_conversion_factor', tool_call_id='f2rk3p9v5')

In [20]:
messages.append(conversion_factor)

In [21]:
messages

[HumanMessage(content='what is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1794182401, "time_last_update_utc": "Mon, 09 Nov 2026 00:00:01 +0000", "time_next_update_unix": 1794268801, "time_next_update_utc": "Tue, 10 Nov 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 94.0567}', name='get_conversion_factor', tool_call_id='f2rk3p9v5')]

In [23]:
result = llm_with_tools.invoke(messages)
result

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'v0mwb02m3', 'function': {'arguments': '{"base_currency_value":10,"conversion_rate":94.0567}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 582, 'total_tokens': 630, 'completion_time': 0.095992037, 'completion_tokens_details': None, 'prompt_time': 0.04414839, 'prompt_tokens_details': None, 'queue_time': 0.051159737, 'total_time': 0.140140427}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1c0a4-d686-7463-9878-a0b470d963cc-0', tool_calls=[{'name': 'convert', 'args': {'base_currency_value': 10, 'conversion_rate': 94.0567}, 'id': 'v0mwb02m3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 582, 'output_tokens': 48, 'total_tokens': 630})

In [24]:
result.tool_calls

[{'name': 'convert',
  'args': {'base_currency_value': 10, 'conversion_rate': 94.0567},
  'id': 'v0mwb02m3',
  'type': 'tool_call'}]

In [25]:
messages.append(result)

In [27]:
final_result = llm_with_tools.invoke(messages)
print(final_result.content)

- Conversion factor (USD → INR): **1 USD = 94.0567 INR**
- Converted amount: **10 USD = 940.567 INR** (rounded to **₹940.57**)
